In [8]:
import zipfile
import os
#POSIX path to the zip file/s
#abfss://WS_BCT_WMPP@onelake.dfs.fabric.microsoft.com/LH_BCT_WMPP.Lakehouse/Files/wmpp-production-data-export-birmingham/archive/2026-06-29.zip
#Files/wmpp-production-data-export-birmingham/archive/2026-06-29.zip
zip_shortcut_path = "/lakehouse/default/Files/wmpp-production-data-export-birmingham/archive/2026-06-29.zip"

# Target path (either in OneLake or local Spark node memory/disk)
extract_path = "/lakehouse/default/Files/archive_unzipped"

os.makedirs(extract_path, exist_ok=True)

# Unzip using standard Python library
with zipfile.ZipFile(zip_shortcut_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Successfully unzipped files from the S3 shortcut!")

StatementMeta(, 8865b6a1-98fd-4730-b086-47d974f1ceef, 10, Finished, Available, Finished, False)

Successfully unzipped files from the S3 shortcut!


In [9]:
## only finds single file in the zip


import zipfile
import os

# 1. Base files directory
base_dir = "/lakehouse/default/Files"

# 2. Automatically find the shortcut directory that starts with "wmpp-production"
shortcut_folder = [f for f in os.listdir(base_dir) if f.startswith("wmpp-production")][0]

# 3. Construct the exact POSIX path to the archive folder
archive_dir = os.path.join(base_dir, shortcut_folder, "archive")
extract_path = os.path.join(base_dir, "archive_unzipped")

os.makedirs(extract_path, exist_ok=True)

# 4. Unzip all daily archives
zip_files = [f for f in os.listdir(archive_dir) if f.endswith('.zip')]
print(f"Found {len(zip_files)} zip files in {archive_dir}")

for filename in zip_files:
    file_path = os.path.join(archive_dir, filename)
    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print(f"Successfully unzipped all files into: {extract_path}")

StatementMeta(, 8865b6a1-98fd-4730-b086-47d974f1ceef, 11, Finished, Available, Finished, False)

Found 31 zip files in /lakehouse/default/Files/wmpp-production-data-export-birmingham/archive
Successfully unzipped all files into: /lakehouse/default/Files/archive_unzipped


In [3]:
import zipfile
import os

# 1. Define paths
base_dir = "/lakehouse/default/Files"

# Automatically find the shortcut folder
shortcut_folder = [f for f in os.listdir(base_dir) if f.startswith("wmpp-production")][0]
archive_dir = os.path.join(base_dir, shortcut_folder, "archive")
extract_base_path = os.path.join(base_dir, "archive_unzipped")

os.makedirs(extract_base_path, exist_ok=True)

# 2. Find all ZIP files
zip_files = [f for f in os.listdir(archive_dir) if f.endswith('.zip')]
print(f"Found {len(zip_files)} ZIP archive(s) to check.\n")

extracted_count = 0
skipped_count = 0

# 3. Loop through each ZIP file
for zip_name in zip_files:
    # Remove .zip extension to get the folder name (e.g., '2026-07-01')
    folder_name = os.path.splitext(zip_name)[0]
    target_folder_path = os.path.join(extract_base_path, folder_name)
    
    # Check if the target folder exists and is non-empty
    if os.path.exists(target_folder_path) and os.listdir(target_folder_path):
        print(f"⏩ Skipped (folder already exists): {folder_name}/")
        skipped_count += 1
    else:
        # Create the specific folder for this zip file
        os.makedirs(target_folder_path, exist_ok=True)
        zip_file_path = os.path.join(archive_dir, zip_name)
        
        # Extract files straight into the date folder
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(target_folder_path)
            
        print(f"✅ Extracted: {zip_name} ➔ {folder_name}/")
        extracted_count += 1

print(f"\nFinished processing! Extracted: {extracted_count} | Skipped: {skipped_count}\n")

# 4. Read ALL CSV files across all date subfolders into PySpark
df_all = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("recursiveFileLookup", "true") # Digs into subfolders (2026-07-01/, 2026-07-02/, etc.)
    .csv("Files/archive_unzipped")
)

print(f"Total rows in consolidated dataset: {df_all.count():,}")
display(df_all)

StatementMeta(, 80df82b9-b269-4955-bf9d-4d95ea1d6255, 4, Finished, Available, Finished, False)

Found 30 ZIP archive(s) to check.

✅ Extracted: 2026-06-30.zip ➔ 2026-06-30/
✅ Extracted: 2026-07-01.zip ➔ 2026-07-01/
✅ Extracted: 2026-07-02.zip ➔ 2026-07-02/
✅ Extracted: 2026-07-03.zip ➔ 2026-07-03/
✅ Extracted: 2026-07-04.zip ➔ 2026-07-04/
✅ Extracted: 2026-07-05.zip ➔ 2026-07-05/
✅ Extracted: 2026-07-06.zip ➔ 2026-07-06/
✅ Extracted: 2026-07-07.zip ➔ 2026-07-07/
✅ Extracted: 2026-07-08.zip ➔ 2026-07-08/
✅ Extracted: 2026-07-09.zip ➔ 2026-07-09/
✅ Extracted: 2026-07-10.zip ➔ 2026-07-10/
✅ Extracted: 2026-07-11.zip ➔ 2026-07-11/
✅ Extracted: 2026-07-12.zip ➔ 2026-07-12/
✅ Extracted: 2026-07-13.zip ➔ 2026-07-13/
✅ Extracted: 2026-07-14.zip ➔ 2026-07-14/
✅ Extracted: 2026-07-15.zip ➔ 2026-07-15/
✅ Extracted: 2026-07-16.zip ➔ 2026-07-16/
✅ Extracted: 2026-07-17.zip ➔ 2026-07-17/
✅ Extracted: 2026-07-18.zip ➔ 2026-07-18/
✅ Extracted: 2026-07-19.zip ➔ 2026-07-19/
✅ Extracted: 2026-07-20.zip ➔ 2026-07-20/
✅ Extracted: 2026-07-21.zip ➔ 2026-07-21/
✅ Extracted: 2026-07-22.zip ➔ 2026-07-22/

SynapseWidget(Synapse.DataFrame, 271dc13c-656a-4b49-8590-8c5e154d8b0e)

In [25]:
df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"Files/archive_unzipped/audit/jun26/2026-06-01.csv")
)

display(df)


StatementMeta(, 8865b6a1-98fd-4730-b086-47d974f1ceef, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b819e0b7-35bf-4c36-8607-bf6d98074e16)

StatementMeta(, 1320649f-8fb5-4796-8b6f-693df0702f0f, 16, Finished, Available, Finished, False)

ROOT_PATH=[Files/archive_unzipped]
BRONZE_SCHEMA=[archived]
TABLE_PREFIX=[archived_]
LOAD_MODE=[append]
TEXT_QUALIFIER=["]
REBUILD=[0]


In [2]:
%%sql
drop schema archived;

--drop table if exists archived.cfg_load_control;
CREATE schema IF NOT EXISTS archived


StatementMeta(, bf094b74-e9b1-4e2d-8216-d3d453c09115, 4, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [59]:
%%sql

drop table archived.archived_provider_submission_docs;

StatementMeta(, 1320649f-8fb5-4796-8b6f-693df0702f0f, 63, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
%%sql 
select * from archived.cfg_load_control order by 

StatementMeta(, bf094b74-e9b1-4e2d-8216-d3d453c09115, 2, Finished, Available, Finished, False)

<Spark SQL result set with 55 rows and 9 fields>

In [3]:
import os, re
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, IntegerType, TimestampType
from delta.tables import DeltaTable

# ── 1. Parameters ─────────────────────────────────────────────────────────────
ROOT_PATH        = "Files/archive_unzipped"      # Relative path inside Lakehouse
ERROR_PATH        = "Files/archive_error_logs/corrupt_rows"      # Malformed path inside Lakehouse
POSIX_ROOT_PATH  = f"/lakehouse/default/{ROOT_PATH}"
ARCHIVE_SCHEMA   = "archived"
TABLE_PREFIX     = "archived_"                  # Prefix for bronze tables
LOAD_MODE        = "append"                     # append | overwrite
TEXT_QUALIFIER   = '"'
REBUILD          = 0                            # Set to 1 to drop & rebuild tables


print(f"ROOT_PATH=[{ROOT_PATH}]")
print(f"ERROR_PATH=[{ERROR_PATH}]")
print(f"ARCHIVE_SCHEMA=[{ARCHIVE_SCHEMA}]")
print(f"TABLE_PREFIX=[{TABLE_PREFIX}]")
print(f"LOAD_MODE=[{LOAD_MODE}]")
print(f"REBUILD=[{REBUILD}]")

# ── 2. Create Schema & Control Table ──────────────────────────────────────────
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ARCHIVE_SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ARCHIVE_SCHEMA}.cfg_load_control 
(
    filename STRING,
    reload BOOLEAN,
    table_path STRING,
    schema_name STRING,
    table_name STRING,
    load_count INT,
    export_date STRING,
    first_load_date TIMESTAMP,
    last_load_date TIMESTAMP
)
""")

# ── 3. Load Previously Logged Paths into Memory ────────────────────────────────
loaded_df = spark.table(f"{ARCHIVE_SCHEMA}.cfg_load_control").where("reload = false").select("table_path")
loaded_paths = set(row["table_path"] for row in loaded_df.collect())

print(f"Found {len(loaded_paths)} previously logged file path(s) in control table.\n")

# ── 4. Recursively Process All Files Across All Date Folders ──────────────────
processed_count = 0
skipped_count = 0

for root, dirs, files in os.walk(POSIX_ROOT_PATH):
    for file_name in files:
        # Only target CSV or Parquet files
        if not (file_name.lower().endswith('.csv') or file_name.lower().endswith('.parquet')):
            continue

        # Paths
        full_posix_path = os.path.join(root, file_name)
        relative_path = os.path.relpath(full_posix_path, "/lakehouse/default")
        
        # Check control table for skip
        if relative_path in loaded_paths:
            print(f"⏩ Skipped (already logged): {relative_path}")
            skipped_count += 1
            continue

        # Extract target table name (e.g. ProviderHome.csv -> archived.archived_ProviderHome)
        raw_entity_name = os.path.splitext(file_name)[0]
        clean_table_name = f"{TABLE_PREFIX}{raw_entity_name}"
        if re.search(r"\d{4}-\d{2}-\d{2}" , clean_table_name) :
            clean_table_name = f"{TABLE_PREFIX}audit"

        full_table_target = f"{ARCHIVE_SCHEMA}.{clean_table_name}"

        print(f"\n🔄 Processing: {relative_path}")
        print(f"   └── Target Table: {full_table_target}")

        # Drop table if REBUILD flag is active
        if REBUILD == 1:
            print(f"\t🔥 REBUILD=1: Dropping table {full_table_target}")
            spark.sql(f"DROP TABLE IF EXISTS {full_table_target}")
        
        ERROR_LOGS = f"Files/{ARCHIVE_SCHEMA}/error_logs"

        # Load file into DataFrame
        try:
            if file_name.lower().endswith('.parquet'):
                df = spark.read.format("parquet").load(relative_path)
            elif file_name.lower().endswith('.csv'):
                df = (
                    spark.read
                    .format("csv")
                    .option("header", "true")
                    .option("inferSchema", "false")
                    .option("mode", "PERMISSIVE")
                    .option("badRecordsPath", ERROR_LOGS)
                    .option("quote", TEXT_QUALIFIER)
                    .option("escape", TEXT_QUALIFIER)
                    .option("multiLine", "true")
                    .load(relative_path)
                )

            row_count = df.count()
            print(f"\tLoaded {row_count:,} rows.")

            # Write DataFrame into Lakehouse Delta table
            df.write.format("delta") \
                .mode(LOAD_MODE) \
                .option("mergeSchema", "true") \
                .saveAsTable(full_table_target)
            
            print(f"\t✅ Written to {full_table_target}")

            # Extract parent folder name as export date (e.g. '2026-07-01')
            parent_folder = os.path.basename(root)
            export_date = parent_folder if parent_folder != "archive_unzipped" else ""
            now = datetime.now()

            # Append metadata record to control table
            log_schema = StructType([
                StructField("filename", StringType(), True),
                StructField("reload", BooleanType(), True),
                StructField("table_path", StringType(), True),
                StructField("schema_name", StringType(), True),
                StructField("table_name", StringType(), True),
                StructField("load_count", IntegerType(), True),
                StructField("export_date", StringType(), True),
                StructField("first_load_date", TimestampType(), True),
                StructField("last_load_date", TimestampType(), True)
            ])

            # 1. Create the single-row logging DataFrame for the current file
            log_data = [(
                file_name,
                False,
                relative_path,
                ARCHIVE_SCHEMA,
                clean_table_name,
                row_count,
                export_date,
                now, # first_load_date
                now  # last_load_date
            )]

            log_df = spark.createDataFrame(log_data, log_schema)

            # 2. Load the Delta table reference
            control_delta_table = DeltaTable.forName(spark, f"{ARCHIVE_SCHEMA}.cfg_load_control")

            # 3. Perform the Upsert (Merge)
            control_delta_table.alias("target") \
                .merge(
                    log_df.alias("source"),
                    "target.table_path = source.table_path"
                ) \
                .whenMatchedUpdate(set = {
                    # Update existing record: add new row_count to previous total
                    "load_count": "target.load_count + source.load_count",
                    "last_load_date": "source.last_load_date" if "source.last_load_date" in log_df.columns else "source.last_load_date"
                }) \
                .whenNotMatchedInsertAll() \
                .execute()

            print(f"\t📝 Logged metadata to {ARCHIVE_SCHEMA}.cfg_load_control")

            # Memory update to prevent duplicate work in current run
            loaded_paths.add(relative_path)
            processed_count += 1

        except Exception as e:
            print(f"❌ FAILED to process {file_name}: {str(e)}")
            raise e

print(f"\n🎉 Completed! Total files processed: {processed_count} | Skipped: {skipped_count}")

StatementMeta(, bf094b74-e9b1-4e2d-8216-d3d453c09115, 6, Submitted, Running, Running, True)

ROOT_PATH=[Files/archive_unzipped]
ERROR_PATH=[Files/archive_error_logs/corrupt_rows]
ARCHIVE_SCHEMA=[archived]
TABLE_PREFIX=[archived_]
LOAD_MODE=[append]
REBUILD=[0]
Found 0 previously logged file path(s) in control table.


🔄 Processing: Files/archive_unzipped/2026-06-30/additional_fee.csv
   └── Target Table: archived.archived_additional_fee
	Loaded 120 rows.
	✅ Written to archived.archived_additional_fee
	📝 Logged metadata to archived.cfg_load_control

🔄 Processing: Files/archive_unzipped/2026-06-30/framework_category.csv
   └── Target Table: archived.archived_framework_category
	Loaded 20 rows.
	✅ Written to archived.archived_framework_category
	📝 Logged metadata to archived.cfg_load_control

🔄 Processing: Files/archive_unzipped/2026-06-30/holding_company.csv
   └── Target Table: archived.archived_holding_company
	Loaded 606 rows.
	✅ Written to archived.archived_holding_company
	📝 Logged metadata to archived.cfg_load_control

🔄 Processing: Files/archive_unzipped/2026-06-30/ipa.cs

In [21]:
df = spark.sql("SELECT * FROM LH_BCT_WMPP.archived.archived_provider_home_spot_category LIMIT 1000")
# display(df)

StatementMeta(, 1320649f-8fb5-4796-8b6f-693df0702f0f, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 76997c7d-a8fb-462e-964d-e6c0cc005d55)

In [ ]:
df = spark.sql("SELECT * FROM LH_BCT_WMPP.archived.archived_audit LIMIT 1000")

display(df.groupBy("change_type").count())


StatementMeta(, bf094b74-e9b1-4e2d-8216-d3d453c09115, -1, Cancelled, , Cancelled, True)

In [ ]:
%%sql
-- -- Claire Differ,ProviderHome,"""1ff342e9-f03b-4729-8b1d-e0ac64793bae""",,CREATED,,"{serviceType=""PLCM-REST""
-- select serviceType, * from bronze.Provider_Home where provider_id ='1ff342e9-f03b-4729-8b1d-e0ac64793bae'

--Provider,"""5475628a-ba0f-4f70-8455-a12f4208b1f6""" "{registrantEmailAddress=""contracts@compasscommunity.co.uk""
-- select * from bronze.provider where provider_id ='5475628a-ba0f-4f70-8455-a12f4208b1f6'
